# Lab 04 — Warehouse の検証（読み取り専用）

T-SQL Notebook `nb_04_build_warehouse` が作成した Warehouse のテーブルを Spark コネクタで読み取り、
件数・キーの整合性・金額が Lakehouse の値と一致するか確認します。Warehouse は変更しません。

In [ ]:
import json
from datetime import datetime, timezone

def save_result(name, payload):
    """Persist a run summary that scripts/lab.py reads back from OneLake."""
    payload = {"notebook": name, "finishedAtUtc": datetime.now(timezone.utc).isoformat(timespec="seconds"), **payload}
    notebookutils.fs.put(f"Files/lab_results/{name}.json", json.dumps(payload, ensure_ascii=False, default=str), True)
    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))

checks = []

def check(name, expected, actual, tolerance=None):
    if tolerance is None:
        passed = expected == actual
    else:
        passed = actual is not None and abs(float(expected) - float(actual)) <= tolerance
    checks.append({"check": name, "expected": expected, "actual": actual, "passed": bool(passed)})

In [ ]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants
from pyspark.sql import functions as F

def wh(table):
    return spark.read.synapsesql(f"taxi_warehouse.dbo.{table}")

EXPECTED_ROWS = {"fact_trips": 37948, "dim_vendor": 2, "dim_ratecode": 6, "dim_payment_type": 6,
                 "dim_trip_type": 2, "dim_pickup_zone": 265, "dim_dropoff_zone": 265}
tables = {name: wh(name).cache() for name in EXPECTED_ROWS}
for name, rows in EXPECTED_ROWS.items():
    check(f"{name} rows", rows, tables[name].count())

fact = tables["fact_trips"]
lakehouse_total = spark.table("green_tripdata_2026").agg(F.round(F.sum("total_amount"), 2)).first()[0]
check("total_amount equals Lakehouse", float(lakehouse_total), float(fact.agg(F.round(F.sum("total_amount"), 2)).first()[0]), 0.005)
check("pickup datetime kept (non-null)", 37948, fact.filter(F.col("lpep_pickup_datetime").isNotNull()).count())

for fact_key, dim, dim_key in [("VendorID", "dim_vendor", "VendorID"), ("RatecodeID", "dim_ratecode", "RatecodeID"),
                               ("payment_type", "dim_payment_type", "payment_type"), ("trip_type", "dim_trip_type", "trip_type"),
                               ("PULocationID", "dim_pickup_zone", "LocationID"), ("DOLocationID", "dim_dropoff_zone", "LocationID")]:
    orphans = fact.join(tables[dim], fact[fact_key] == tables[dim][dim_key], "left_anti").count()
    check(f"orphan keys {fact_key} -> {dim}", 0, orphans)
    duplicates = tables[dim].groupBy(dim_key).count().filter("count > 1").count()
    check(f"duplicate keys in {dim}", 0, duplicates)

mb = (fact.join(tables["dim_pickup_zone"], fact.PULocationID == tables["dim_pickup_zone"].LocationID)
          .join(tables["dim_dropoff_zone"], fact.DOLocationID == tables["dim_dropoff_zone"].LocationID)
          .filter((F.col("pickup_borough") == "Manhattan") & (F.col("dropoff_borough") == "Brooklyn")))
check("Manhattan -> Brooklyn trips", 2340, mb.count())

failed = [c for c in checks if not c["passed"]]
save_result("nb_04_validate_warehouse", {"passed": not failed, "failed": len(failed), "checks": checks})